# Pandas-SQL Ödevi Çözüm Notebook'u

Bu notebook, SQLite Sakila film kiralama veritabanındaki SQL sorularının Pandas karşılıklarını içerir.




In [3]:
import sqlite3
import pandas as pd
import numpy as np

DB_PATH = "sqlite-sakila.db"  
conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name;
""", conn)
tables


C:\Users\hp\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,name
0,actor
1,address
2,category
3,city
4,country
5,customer
6,film
7,film_actor
8,film_category
9,film_text


In [4]:

actor = pd.read_sql_query("SELECT * FROM actor", conn)
address = pd.read_sql_query("SELECT * FROM address", conn)
category = pd.read_sql_query("SELECT * FROM category", conn)
city = pd.read_sql_query("SELECT * FROM city", conn)
country = pd.read_sql_query("SELECT * FROM country", conn)
customer = pd.read_sql_query("SELECT * FROM customer", conn)
film = pd.read_sql_query("SELECT * FROM film", conn)
film_actor = pd.read_sql_query("SELECT * FROM film_actor", conn)
film_category = pd.read_sql_query("SELECT * FROM film_category", conn)
inventory = pd.read_sql_query("SELECT * FROM inventory", conn)
language = pd.read_sql_query("SELECT * FROM language", conn)
payment = pd.read_sql_query("SELECT * FROM payment", conn)
rental = pd.read_sql_query("SELECT * FROM rental", conn)
staff = pd.read_sql_query("SELECT * FROM staff", conn)
store = pd.read_sql_query("SELECT * FROM store", conn)


In [5]:

actor_df = actor.copy()
actor_df["actor_name"] = actor_df["first_name"] + " " + actor_df["last_name"]

customer_df = customer.copy()
customer_df["customer_name"] = customer_df["first_name"] + " " + customer_df["last_name"]

staff_df = staff.copy()
staff_df["staff_name"] = staff_df["first_name"] + " " + staff_df["last_name"]

customer_location = (
    customer_df[["customer_id", "customer_name", "store_id", "address_id"]]
    .merge(address[["address_id", "city_id"]], on="address_id", how="left")
    .merge(city[["city_id", "city", "country_id"]], on="city_id", how="left")
    .merge(country[["country_id", "country"]], on="country_id", how="left")
)

store_location = (
    store[["store_id", "address_id"]]
    .merge(address[["address_id", "city_id"]], on="address_id", how="left")
    .merge(city[["city_id", "city", "country_id"]], on="city_id", how="left")
    .merge(country[["country_id", "country"]], on="country_id", how="left")
    .rename(columns={"city": "store_city", "country": "store_country"})
)

film_cat = (
    film[["film_id", "title", "rental_duration", "rental_rate", "length"]]
    .merge(film_category, on="film_id", how="left")
    .merge(category.rename(columns={"name": "category_name"})[["category_id", "category_name"]], on="category_id", how="left")
)

rental_full = (
    rental
    .merge(inventory[["inventory_id", "film_id", "store_id"]], on="inventory_id", how="left")
    .merge(film[["film_id", "title", "rental_duration", "rental_rate", "length"]], on="film_id", how="left")
    .merge(payment[["payment_id", "rental_id", "amount", "payment_date"]], on="rental_id", how="left")
    .merge(customer_df[["customer_id", "customer_name"]], on="customer_id", how="left")
    .merge(staff_df[["staff_id", "staff_name"]], on="staff_id", how="left")
    .merge(film_category, on="film_id", how="left")
    .merge(category.rename(columns={"name": "category_name"})[["category_id", "category_name"]], on="category_id", how="left")
    .merge(customer_location[["customer_id", "city", "country"]], on="customer_id", how="left")
    .merge(store_location[["store_id", "store_city", "store_country"]], on="store_id", how="left")
)

# Tarih ve süre kolonları
rental_full["rental_date"] = pd.to_datetime(rental_full["rental_date"])
rental_full["return_date"] = pd.to_datetime(rental_full["return_date"])
rental_full["payment_date"] = pd.to_datetime(rental_full["payment_date"])
rental_full["rental_days"] = (rental_full["return_date"] - rental_full["rental_date"]).dt.total_seconds() / (60 * 60 * 24)
rental_full["due_date"] = rental_full["rental_date"] + pd.to_timedelta(rental_full["rental_duration"], unit="D")



## Soru 1: Tüm verileri göster


In [6]:
# Örnek olarak film tablosunun tüm verileri
film


,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,None,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2021-03-06 15:52:00
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,1,None,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2021-03-06 15:52:00
2,3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a ...,2006,1,None,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes",2021-03-06 15:52:00
3,4,AFFAIR PREJUDICE,A Fanciful Documentary of a Frisbee And a Lumb...,2006,1,None,5,2.99,117,26.99,G,"Commentaries,Behind the Scenes",2021-03-06 15:52:00
4,5,AFRICAN EGG,A Fast-Paced Documentary of a Pastry Chef And ...,2006,1,None,6,2.99,130,22.99,G,Deleted Scenes,2021-03-06 15:52:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,YOUNG LANGUAGE,A Unbelieveable Yarn of a Boat And a Database ...,2006,1,None,6,0.99,183,9.99,G,"Trailers,Behind the Scenes",2021-03-06 15:52:08
996,997,YOUTH KICK,A Touching Drama of a Teacher And a Cat who mu...,2006,1,None,4,0.99,179,14.99,NC-17,"Trailers,Behind the Scenes",2021-03-06 15:52:08
997,998,ZHIVAGO CORE,A Fateful Yarn of a Composer And a Man who mus...,2006,1,None,6,0.99,105,10.99,NC-17,Deleted Scenes,2021-03-06 15:52:08
998,999,ZOOLANDER FICTION,A Fateful Reflection of a Waitress And a Boat ...,2006,1,None,5,2.99,101,28.99,R,"Trailers,Deleted Scenes",2021-03-06 15:52:08


## Soru 2: Her filmdeki oyuncuları listele


In [7]:
soru2 = (
    film[["film_id", "title"]]
    .merge(film_actor, on="film_id")
    .merge(actor_df[["actor_id", "actor_name"]], on="actor_id")
    [["title", "actor_name"]]
    .sort_values(["title", "actor_name"])
)
soru2


,title,actor_name
1,ACADEMY DINOSAUR,CHRISTIAN GABLE
4,ACADEMY DINOSAUR,JOHNNY CAGE
2,ACADEMY DINOSAUR,LUCILLE TRACY
9,ACADEMY DINOSAUR,MARY KEITEL
5,ACADEMY DINOSAUR,MENA TEMPLE
...,...,...
5456,ZOOLANDER FICTION,PENELOPE CRONYN
5457,ZOOLANDER FICTION,WHOOPI HURT
5459,ZORRO ARK,IAN TANDY
5461,ZORRO ARK,LISA MONROE


## Soru 3: Her filmde kaç oyuncu oynadı?


In [ ]:
soru3 = (
    film[["film_id", "title"]]
    .merge(film_actor, on="film_id")
    .groupby(["film_id", "title"])["actor_id"]
    .nunique()
    .reset_index(name="actor_count")
    .sort_values("actor_count", ascending=False)
)
soru3


## Soru 4: Her oyuncu kaç filmde oynadı?


In [8]:
soru4 = (
    actor_df[["actor_id", "actor_name"]]
    .merge(film_actor, on="actor_id")
    .groupby(["actor_id", "actor_name"])["film_id"]
    .nunique()
    .reset_index(name="film_count")
    .sort_values("film_count", ascending=False)
)
soru4


,actor_id,actor_name,film_count
106,107,GINA DEGENERES,42
101,102,WALTER TORN,41
197,198,MARY KEITEL,40
180,181,MATTHEW CARREY,39
22,23,SANDRA KILMER,37
...,...,...,...
70,71,ADAM GRANT,18
185,186,JULIA ZELLWEGER,16
34,35,JUDY DEAN,15
198,199,JULIA FAWCETT,15


## Soru 5: Envanterde olmayan filmler var mı ve varsa kaç tane?


In [9]:
soru5_filmler = (
    film[["film_id", "title"]]
    .merge(inventory[["film_id"]].drop_duplicates(), on="film_id", how="left", indicator=True)
    .query("_merge == 'left_only'")
    [["film_id", "title"]]
)

soru5_sayi = soru5_filmler["film_id"].nunique()
soru5_filmler, soru5_sayi


(     film_id                   title
 13        14          ALICE FANTASIA
 32        33             APOLLO TEEN
 35        36          ARGONAUTS TOWN
 37        38           ARK RIDGEMONT
 40        41    ARSENIC INDEPENDENCE
 86        87       BOONDOCK BALLROOM
 107      108           BUTCH PANTHER
 127      128           CATCH AMISTAD
 143      144     CHINATOWN GLADIATOR
 147      148          CHOCOLATE DUCK
 170      171    COMMANDMENTS EXPRESS
 191      192        CROSSING DIVORCE
 194      195         CROWDS TELEMARK
 197      198        CRYSTAL BREAKING
 216      217              DAZED PUNK
 220      221  DELIVERANCE MULHOLLAND
 317      318       FIREHOUSE VIETNAM
 324      325           FLOATS GARDEN
 331      332   FRANKENSTEIN STRANGER
 358      359      GLADIATOR WESTWARD
 385      386               GUMP DATE
 403      404           HATE HANDICAP
 418      419             HOCUS FRIDA
 494      495        KENTUCKIAN GIANT
 496      497        KILL BROTHERHOOD
 606      60

## Soru 6: Kiralanabilir olan her filmin kaç kez kiralandığını ve toplam gelirlerini getirin


In [10]:
soru6 = (
    rental_full.dropna(subset=["film_id"])
    .groupby(["film_id", "title"])
    .agg(rental_count=("rental_id", "nunique"), total_revenue=("amount", "sum"))
    .reset_index()
    .sort_values(["rental_count", "total_revenue"], ascending=False)
)
soru6


,film_id,title,rental_count,total_revenue
96,103,BUCKET BROTHERHOOD,34,180.66
705,738,ROCKETEER MOTHER,33,116.67
733,767,SCALAWAG DUCK,32,172.68
697,730,RIDGEMONT SUBMARINE,32,130.68
312,331,FORWARD TEMPLE,32,128.68
...,...,...,...,...
341,362,GLORY TRACY,5,14.95
315,335,FREEDOM CLEOPATRA,5,5.95
866,904,TRAIN BUNCH,4,24.96
378,400,HARDLY ROBBERS,4,15.96


## Soru 7: Envanterde olmayan filmlerin kira oranlarını getirin


In [11]:
soru7 = (
    film[["film_id", "title", "rental_rate"]]
    .merge(inventory[["film_id"]].drop_duplicates(), on="film_id", how="left", indicator=True)
    .query("_merge == 'left_only'")
    [["film_id", "title", "rental_rate"]]
    .sort_values("rental_rate", ascending=False)
)
soru7


,film_id,title,rental_rate
954,955,WALLS ARTIST,4.99
191,192,CROSSING DIVORCE,4.99
358,359,GLADIATOR WESTWARD,4.99
800,801,SISTER FREDDY,4.99
712,713,RAINBOW SHOCK,4.99
216,217,DAZED PUNK,4.99
606,607,MUPPET MILE,4.99
194,195,CROWDS TELEMARK,4.99
170,171,COMMANDMENTS EXPRESS,4.99
143,144,CHINATOWN GLADIATOR,4.99


## Soru 8: Birden fazla DVD'yi iade etmeyen kaç müşteri var?


In [12]:
not_returned = rental[rental["return_date"].isna()]
soru8_liste = (
    not_returned
    .groupby("customer_id")["rental_id"]
    .nunique()
    .reset_index(name="not_returned_count")
    .query("not_returned_count > 1")
    .merge(customer_df[["customer_id", "customer_name"]], on="customer_id", how="left")
)
soru8_sayi = soru8_liste["customer_id"].nunique()
soru8_liste, soru8_sayi


(    customer_id  not_returned_count      customer_name
 0            15                   2       HELEN HARRIS
 1            42                   2      CAROLYN PEREZ
 2            43                   2  CHRISTINE ROBERTS
 3            53                   2     HEATHER MORRIS
 4            60                   2     MILDRED BAILEY
 5            75                   3      TAMMY SANDERS
 6           107                   2     FLORENCE WOODS
 7           155                   2        GAIL KNIGHT
 8           163                   2      CATHY SPENCER
 9           175                   2      ANNETTE OLSON
 10          208                   2       LUCY WHEELER
 11          216                   2      NATALIE MEYER
 12          228                   2    ALLISON STANLEY
 13          267                   2        MARGIE WADE
 14          269                   2  CASSANDRA WALTERS
 15          284                   2      SONIA GREGORY
 16          354                   2         JUS

## Soru 9: Her müşteri kaç film kiraladı?


In [13]:
soru9 = (
    rental_full
    .groupby(["customer_id", "customer_name"])["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
    .sort_values("rental_count", ascending=False)
)
soru9


,customer_id,customer_name,rental_count
147,148,ELEANOR HUNT,46
525,526,KARL SEAL,45
143,144,CLARA SHAW,42
235,236,MARCIA DEAN,42
74,75,TAMMY SANDERS,41
...,...,...,...
247,248,CAROLINE BOWMAN,15
109,110,TIFFANY JORDAN,14
60,61,KATHERINE RIVERA,14
280,281,LEONA OBRIEN,14


## Soru 10: Türlerine göre en çok kiralanan filmler ve bunlara ne kadar ödendi?


In [14]:
soru10 = (
    rental_full
    .groupby(["category_name", "film_id", "title"])
    .agg(rental_count=("rental_id", "nunique"), total_paid=("amount", "sum"))
    .reset_index()
    .sort_values(["category_name", "rental_count"], ascending=[True, False])
)
soru10


,category_name,film_id,title,rental_count,total_paid
46,Action,748,RUGRATS SHAKESPEARE,30,70.70
53,Action,869,SUSPECTS QUILLS,30,133.70
29,Action,395,HANDICAP BOONDOCK,28,63.72
52,Action,850,STORY SIDE,28,39.72
54,Action,911,TRIP NEWTON,28,145.72
...,...,...,...,...,...
911,Travel,125,CASSIDY WYOMING,6,23.94
915,Travel,224,DESPERATE TRAINSPOTTING,6,31.94
926,Travel,405,HAUNTED ANTITRUST,6,31.94
929,Travel,472,ITALIAN AFRICAN,6,38.94


## Soru 11: Tür ve Tarihe Göre Kiralama Sayısı ve Gelir


In [15]:
soru11 = rental_full.copy()
soru11["rental_day"] = soru11["rental_date"].dt.date
soru11 = (
    soru11
    .groupby(["category_name", "rental_day"])
    .agg(rental_count=("rental_id", "nunique"), total_revenue=("amount", "sum"))
    .reset_index()
    .sort_values(["category_name", "rental_day"])
)
soru11


,category_name,rental_day,rental_count,total_revenue
0,Action,2005-05-25,10,46.90
1,Action,2005-05-26,15,80.85
2,Action,2005-05-27,9,37.91
3,Action,2005-05-28,20,68.80
4,Action,2005-05-29,12,42.88
...,...,...,...,...
628,Travel,2005-08-20,29,127.71
629,Travel,2005-08-21,39,163.61
630,Travel,2005-08-22,21,103.79
631,Travel,2005-08-23,27,126.73


## Soru 12: Kiralanabilir Filmler İçin Türlerine Göre Her Filmin Kaç Kez Kiralandığı


In [16]:
soru12 = (
    rental_full
    .groupby(["category_name", "film_id", "title"])["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
    .sort_values(["category_name", "rental_count"], ascending=[True, False])
)
soru12


,category_name,film_id,title,rental_count
46,Action,748,RUGRATS SHAKESPEARE,30
53,Action,869,SUSPECTS QUILLS,30
29,Action,395,HANDICAP BOONDOCK,28
52,Action,850,STORY SIDE,28
54,Action,911,TRIP NEWTON,28
...,...,...,...,...
911,Travel,125,CASSIDY WYOMING,6
915,Travel,224,DESPERATE TRAINSPOTTING,6
926,Travel,405,HAUNTED ANTITRUST,6
929,Travel,472,ITALIAN AFRICAN,6


## Soru 13: En çok rafta bekleyen filmler


In [17]:
# Yorum: En az kiralanan / hiç kiralanmayan envanterdeki filmler rafta daha çok beklemiş kabul edilir.
soru13 = (
    inventory[["inventory_id", "film_id"]]
    .merge(film[["film_id", "title"]], on="film_id", how="left")
    .merge(rental[["inventory_id", "rental_id"]], on="inventory_id", how="left")
    .groupby(["film_id", "title"])
    .agg(inventory_count=("inventory_id", "nunique"), rental_count=("rental_id", "nunique"))
    .reset_index()
    .sort_values(["rental_count", "inventory_count"], ascending=[True, False])
)
soru13


,film_id,title,inventory_count,rental_count
378,400,HARDLY ROBBERS,2,4
558,584,MIXED DOORS,2,4
866,904,TRAIN BUNCH,2,4
87,94,BRAVEHEART HUMAN,2,5
100,107,BUNCH MINDS,2,5
...,...,...,...,...
465,489,JUGGLER HARDLY,8,32
697,730,RIDGEMONT SUBMARINE,8,32
733,767,SCALAWAG DUCK,8,32
705,738,ROCKETEER MOTHER,8,33


## Soru 14: Geç, Erken ve Zamanında İade Edilen Kiralanmış Filmler


In [18]:
soru14 = rental_full.dropna(subset=["return_date"]).copy()
soru14["return_status"] = np.select(
    [soru14["return_date"] > soru14["due_date"], soru14["return_date"] < soru14["due_date"]],
    ["Geç", "Erken"],
    default="Zamanında"
)
soru14 = (
    soru14
    .groupby("return_status")["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
)
soru14


,return_status,rental_count
0,Erken,7738
1,Geç,8121
2,Zamanında,2


## Soru 15: Hangi müşteri en çok DVD kiralamış?


In [19]:
soru15 = (
    rental_full
    .groupby(["customer_id", "customer_name"])["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
    .sort_values("rental_count", ascending=False)
    .head(1)
)
soru15


,customer_id,customer_name,rental_count
147,148,ELEANOR HUNT,46


## Soru 16: En popüler film kategorisi nedir?


In [20]:
soru16 = (
    rental_full
    .groupby("category_name")["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
    .sort_values("rental_count", ascending=False)
    .head(1)
)
soru16


,category_name,rental_count
14,Sports,1179


## Soru 17: Hangi çalışan en çok kiralama işlemi gerçekleştirmiş?


In [22]:
soru17 = (
    rental_full
    .groupby(["staff_id", "staff_name"])["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
    .sort_values("rental_count", ascending=False)
    .head(1)
)
soru17


,staff_id,staff_name,rental_count
0,1,Mike Hillyer,8040


## Soru 18: En çok geliri hangi film getirmiş?


In [23]:
soru18 = (
    rental_full
    .groupby(["film_id", "title"])["amount"]
    .sum()
    .reset_index(name="total_revenue")
    .sort_values("total_revenue", ascending=False)
    .head(1)
)
soru18


,film_id,title,total_revenue
841,879,TELEGRAPH VOYAGE,231.73


## Soru 19: Her müşteri için toplam harcama miktarını bulun


In [24]:
soru19 = (
    rental_full
    .groupby(["customer_id", "customer_name"])["amount"]
    .sum()
    .reset_index(name="total_spending")
    .sort_values("total_spending", ascending=False)
)
soru19


,customer_id,customer_name,total_spending
525,526,KARL SEAL,221.55
147,148,ELEANOR HUNT,216.54
143,144,CLARA SHAW,195.58
136,137,RHONDA KENNEDY,194.61
177,178,MARION SNYDER,194.61
...,...,...,...
96,97,ANNIE RUSSELL,58.82
394,395,JOHNNY TURPIN,57.81
317,318,BRIAN WYMAN,52.88
280,281,LEONA OBRIEN,50.86


## Soru 20: Her kategorideki toplam kiralama sayısını ve gelirleri bulun


In [25]:
soru20 = (
    rental_full
    .groupby("category_name")
    .agg(total_rentals=("rental_id", "nunique"), total_revenue=("amount", "sum"))
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
soru20


,category_name,total_rentals,total_revenue
14,Sports,1179,5314.21
13,Sci-Fi,1101,4756.98
1,Animation,1166,4656.30
6,Drama,1060,4587.39
4,Comedy,941,4383.58
0,Action,1112,4375.85
12,New,940,4351.62
9,Games,969,4281.33
8,Foreign,1033,4270.67
7,Family,1096,4226.07


## Soru 21: En uzun süre kirada kalmış filmleri bulun


In [26]:
soru21 = (
    rental_full.dropna(subset=["rental_days"])
    .groupby(["film_id", "title"])["rental_days"]
    .max()
    .reset_index(name="max_rental_days")
    .sort_values("max_rental_days", ascending=False)
)
soru21


,film_id,title,max_rental_days
400,424,HOLOCAUST HIGHBALL,9.249306
393,416,HIGHBALL POTTER,9.249306
160,172,CONEHEADS SMOOCHY,9.248611
625,653,PANIC CLUB,9.248611
535,561,MASK PEACH,9.248611
...,...,...,...
800,836,SQUAD FISH,5.908333
168,180,CONSPIRACY SPIRIT,5.809722
341,362,GLORY TRACY,5.120833
865,903,TRAFFIC HOBBIT,4.977778


## Soru 22: En az kiralanan 5 film hangileridir?


In [27]:
soru22 = (
    film[["film_id", "title"]]
    .merge(rental_full.groupby("film_id")["rental_id"].nunique().reset_index(name="rental_count"), on="film_id", how="left")
    .fillna({"rental_count": 0})
    .sort_values("rental_count", ascending=True)
    .head(5)
)
soru22


,film_id,title,rental_count
127,128,CATCH AMISTAD,0.0
331,332,FRANKENSTEIN STRANGER,0.0
641,642,ORDER BETRAYED,0.0
859,860,SUICIDES SILENCE,0.0
741,742,ROOF CHAMPION,0.0


## Soru 24: En fazla kazanç sağlayan 5 müşteriyi bulun


In [28]:
soru24 = (
    rental_full
    .groupby(["customer_id", "customer_name"])["amount"]
    .sum()
    .reset_index(name="total_revenue")
    .sort_values("total_revenue", ascending=False)
    .head(5)
)
soru24


,customer_id,customer_name,total_revenue
525,526,KARL SEAL,221.55
147,148,ELEANOR HUNT,216.54
143,144,CLARA SHAW,195.58
136,137,RHONDA KENNEDY,194.61
177,178,MARION SNYDER,194.61


## Soru 25: Her filmin ortalama kiralanma süresini bulun


In [29]:
soru25 = (
    rental_full.dropna(subset=["rental_days"])
    .groupby(["film_id", "title"])["rental_days"]
    .mean()
    .reset_index(name="avg_rental_days")
    .sort_values("avg_rental_days", ascending=False)
)
soru25


,film_id,title,avg_rental_days
305,323,FLIGHT LIES,7.270775
430,454,IMPACT ALADDIN,7.200617
4,5,AFRICAN EGG,7.106629
378,400,HARDLY ROBBERS,6.948438
520,546,MADRE GABLES,6.811806
...,...,...,...
847,885,TEXAS WATCH,3.016667
754,788,SHIP WONDERLAND,2.986296
315,335,FREEDOM CLEOPATRA,2.959722
453,477,JAWBREAKER BROOKLYN,2.585764


## Soru 26: Her türde en popüler filmi bulun


In [30]:
tmp = (
    rental_full
    .groupby(["category_name", "film_id", "title"])["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
)
soru26 = tmp.loc[tmp.groupby("category_name")["rental_count"].idxmax()].sort_values("category_name")
soru26


,category_name,film_id,title,rental_count
46,Action,748,RUGRATS SHAKESPEARE,30
93,Animation,489,JUGGLER HARDLY,32
165,Children,735,ROBBERS JOON,31
231,Classics,891,TIMBERLAND SKY,31
292,Comedy,1000,ZORRO ARK,31
353,Documentary,973,WIFE TURN,31
380,Drama,418,HOBBIT ALIEN,31
418,Family,31,APACHE DIVINE,31
536,Foreign,738,ROCKETEER MOTHER,33
568,Games,331,FORWARD TEMPLE,32


## Soru 27: Her türde en fazla gelir sağlayan filmi bulun


In [31]:
tmp = (
    rental_full
    .groupby(["category_name", "film_id", "title"])["amount"]
    .sum()
    .reset_index(name="total_revenue")
)
soru27 = tmp.loc[tmp.groupby("category_name")["total_revenue"].idxmax()].sort_values("category_name")
soru27


,category_name,film_id,title,total_revenue
24,Action,327,FOOL MOCKINGBIRD,175.77
74,Animation,239,DOGMA FAMILY,178.70
125,Children,48,BACKLASH UNDEFEATED,158.81
229,Classics,843,STEEL SANTA,141.77
292,Comedy,1000,ZORRO ARK,214.69
353,Documentary,973,WIFE TURN,223.69
408,Drama,897,TORQUE BOUND,198.72
467,Family,715,RANGE MOONWALKER,179.73
514,Foreign,460,INNOCENT USUAL,191.74
584,Games,563,MASSACRE USUAL,179.70


## Soru 28: En çok DVD iade etmeyen müşteriyi bulun


In [32]:
soru28 = (
    rental_full[rental_full["return_date"].isna()]
    .groupby(["customer_id", "customer_name"])["rental_id"]
    .nunique()
    .reset_index(name="not_returned_count")
    .sort_values("not_returned_count", ascending=False)
    .head(1)
)
soru28


,customer_id,customer_name,not_returned_count
23,75,TAMMY SANDERS,3


## Soru 29: En fazla kiralama yapan 5 çalışanı bulun


In [33]:
soru29 = (
    rental_full
    .groupby(["staff_id", "staff_name"])["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
    .sort_values("rental_count", ascending=False)
    .head(5)
)
soru29


,staff_id,staff_name,rental_count
0,1,Mike Hillyer,8040
1,2,Jon Stephens,8004


## Soru 30: En fazla kiralama yapan 5 müşteri hangi şubeden kiralama yapmış?


In [34]:
top5_customers = (
    rental_full.groupby(["customer_id", "customer_name"])["rental_id"]
    .nunique().reset_index(name="rental_count")
    .sort_values("rental_count", ascending=False).head(5)
)
soru30 = top5_customers.merge(customer_location[["customer_id", "store_id"]], on="customer_id", how="left")
soru30


,customer_id,customer_name,rental_count,store_id
0,148,ELEANOR HUNT,46,1
1,526,KARL SEAL,45,2
2,144,CLARA SHAW,42,1
3,236,MARCIA DEAN,42,1
4,75,TAMMY SANDERS,41,2


## Soru 31: Her türde en az kiralanan filmi bulun


In [35]:
tmp = (
    film_cat[["category_name", "film_id", "title"]]
    .merge(rental_full.groupby("film_id")["rental_id"].nunique().reset_index(name="rental_count"), on="film_id", how="left")
    .fillna({"rental_count": 0})
)
soru31 = tmp.loc[tmp.groupby("category_name")["rental_count"].idxmin()].sort_values("category_name")
soru31


,category_name,film_id,title,rental_count
37,Action,38,ARK RIDGEMONT,0.0
35,Animation,36,ARGONAUTS TOWN,0.0
800,Children,801,SISTER FREDDY,0.0
13,Classics,14,ALICE FANTASIA,0.0
331,Comedy,332,FRANKENSTEIN STRANGER,0.0
220,Documentary,221,DELIVERANCE MULHOLLAND,0.0
32,Drama,33,APOLLO TEEN,0.0
358,Family,359,GLADIATOR WESTWARD,0.0
127,Foreign,128,CATCH AMISTAD,0.0
216,Games,217,DAZED PUNK,0.0


## Soru 32: En çok kiralama yapan 5 müşteri hangi şehirde?


In [36]:
top5_customers = (
    rental_full.groupby(["customer_id", "customer_name"])["rental_id"]
    .nunique().reset_index(name="rental_count")
    .sort_values("rental_count", ascending=False).head(5)
)
soru32 = top5_customers.merge(customer_location[["customer_id", "city", "country"]], on="customer_id", how="left")
soru32


,customer_id,customer_name,rental_count,city,country
0,148,ELEANOR HUNT,46,Saint-Denis,Runion
1,526,KARL SEAL,45,Cape Coral,United States
2,144,CLARA SHAW,42,Molodetno,Belarus
3,236,MARCIA DEAN,42,Tanza,Philippines
4,75,TAMMY SANDERS,41,Changhwa,Taiwan


## Soru 33: En çok kazanç sağlayan 5 müşteriyi hangi şehirde bulun?


In [37]:
top5_revenue_customers = (
    rental_full.groupby(["customer_id", "customer_name"])["amount"]
    .sum().reset_index(name="total_revenue")
    .sort_values("total_revenue", ascending=False).head(5)
)
soru33 = top5_revenue_customers.merge(customer_location[["customer_id", "city", "country"]], on="customer_id", how="left")
soru33


,customer_id,customer_name,total_revenue,city,country
0,526,KARL SEAL,221.55,Cape Coral,United States
1,148,ELEANOR HUNT,216.54,Saint-Denis,Runion
2,144,CLARA SHAW,195.58,Molodetno,Belarus
3,137,RHONDA KENNEDY,194.61,Apeldoorn,Netherlands
4,178,MARION SNYDER,194.61,Santa Brbara dOeste,Brazil


## Soru 34: En çok kiralanan 5 filmi hangi şehirde bulun?


In [38]:
soru34 = (
    rental_full
    .groupby(["film_id", "title", "city"])["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
    .sort_values("rental_count", ascending=False)
    .head(5)
)
soru34


,film_id,title,city,rental_count
1735,111,CADDYSHACK JEDI,Sorocaba,3
3491,228,DETECTIVE VISION,Kurgan,3
5091,322,FLATLINERS KILLER,Trshavn,3
3570,233,DISCIPLE MOTHER,Lima,3
11693,741,ROMAN PUNK,Tandil,2


## Soru 35: En az kiralanan 5 filmi hangi şehirde bulun?


In [39]:
soru35 = (
    rental_full
    .groupby(["film_id", "title", "city"])["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
    .sort_values("rental_count", ascending=True)
    .head(5)
)
soru35


,film_id,title,city,rental_count
0,1,ACADEMY DINOSAUR,Almirante Brown,1
10509,666,PAYCHECK WAIT,Lausanne,1
10510,666,PAYCHECK WAIT,Lublin,1
10511,666,PAYCHECK WAIT,Naju,1
10512,666,PAYCHECK WAIT,Sambhal,1


## Soru 36: En çok kazanç sağlayan 5 filmi hangi şehirde bulun?


In [40]:
soru36 = (
    rental_full
    .groupby(["film_id", "title", "city"])["amount"]
    .sum()
    .reset_index(name="total_revenue")
    .sort_values("total_revenue", ascending=False)
    .head(5)
)
soru36


,film_id,title,city,total_revenue
9152,579,MINDS TRUMAN,Probolinggo,18.98
1766,113,CALIFORNIA BIRDS,Plock,18.98
11745,745,ROSES TREASURE,Kamyin,17.98
15394,973,WIFE TURN,Santa Brbara dOeste,17.98
15353,972,WHISPERER GIANT,Bijapur,16.98


## Soru 37: En az kazanç sağlayan 5 filmi hangi şehirde bulun?


In [41]:
soru37 = (
    rental_full
    .groupby(["film_id", "title", "city"])["amount"]
    .sum()
    .reset_index(name="total_revenue")
    .sort_values("total_revenue", ascending=True)
    .head(5)
)
soru37


,film_id,title,city,total_revenue
6226,391,HALF OUTFIELD,Yaound,0.0
3377,220,DEER VIRGINIAN,Allappuzha (Alleppey),0.0
10631,676,PHILADELPHIA WIFE,Lengshuijiang,0.0
1246,79,BLADE POLISH,Merlo,0.0
13275,842,STATE WASTELAND,Erlangen,0.0


## Soru 38: En fazla kiralama yapan müşteri hangi filmleri kiralamış?


In [42]:
top_customer_id = rental_full.groupby("customer_id")["rental_id"].nunique().idxmax()
soru38 = (
    rental_full[rental_full["customer_id"] == top_customer_id]
    [["customer_id", "customer_name", "film_id", "title", "rental_date"]]
    .drop_duplicates()
    .sort_values("rental_date")
)
soru38


,customer_id,customer_name,film_id,title,rental_date
680,148,ELEANOR HUNT,694,PREJUDICE OLEANDER,2005-05-28 23:53:18
1499,148,ELEANOR HUNT,387,GUN BONNIE,2005-06-15 22:02:35
1515,148,ELEANOR HUNT,815,SNATCHERS MONTEZUMA,2005-06-15 23:20:26
2748,148,ELEANOR HUNT,285,ENGLISH BULWORTH,2005-06-19 16:39:23
2840,148,ELEANOR HUNT,331,FORWARD TEMPLE,2005-06-19 22:36:39
2844,148,ELEANOR HUNT,275,EGYPT TENENBAUMS,2005-06-19 22:54:01
3650,148,ELEANOR HUNT,552,MAJESTIC FLOATS,2005-07-06 07:45:13
4077,148,ELEANOR HUNT,301,FAMILY SWEET,2005-07-07 05:09:54
4743,148,ELEANOR HUNT,37,ARIZONA BANG,2005-07-08 13:47:55
4947,148,ELEANOR HUNT,724,REMEMBER DIARY,2005-07-08 22:58:07


## Soru 40: En çok kazanç sağlayan müşteri hangi filmleri kiralamış?


In [43]:
top_customer_id = rental_full.groupby("customer_id")["amount"].sum().idxmax()
soru40 = (
    rental_full[rental_full["customer_id"] == top_customer_id]
    [["customer_id", "customer_name", "film_id", "title", "amount", "rental_date"]]
    .drop_duplicates()
    .sort_values("rental_date")
)
soru40


,customer_id,customer_name,film_id,title,amount,rental_date
493,526,KARL SEAL,226,DESTINY SATURDAY,4.99,2005-05-28 00:40:48
677,526,KARL SEAL,201,CYCLONE FAMILY,4.99,2005-05-28 23:24:57
1013,526,KARL SEAL,810,SLUMS DUCK,2.99,2005-05-31 02:44:57
1253,526,KARL SEAL,313,FIDELITY DEVIL,4.99,2005-06-15 06:13:45
1846,526,KARL SEAL,832,SPLASH GUMP,0.99,2005-06-17 00:07:07
1863,526,KARL SEAL,583,MISSION ZOOLANDER,7.99,2005-06-17 01:49:36
1970,526,KARL SEAL,605,MULHOLLAND BEAST,2.99,2005-06-17 09:25:49
1979,526,KARL SEAL,698,PRINCESS GIANT,2.99,2005-06-17 10:03:34
2395,526,KARL SEAL,658,PARIS WEEKEND,4.99,2005-06-18 15:56:53
2825,526,KARL SEAL,709,RACER EGG,2.99,2005-06-19 20:51:33


## Soru 41: En az kazanç sağlayan müşteri hangi filmleri kiralamış?


In [44]:
low_customer_id = rental_full.groupby("customer_id")["amount"].sum().idxmin()
soru41 = (
    rental_full[rental_full["customer_id"] == low_customer_id]
    [["customer_id", "customer_name", "film_id", "title", "amount", "rental_date"]]
    .drop_duplicates()
    .sort_values("rental_date")
)
soru41


,customer_id,customer_name,film_id,title,amount,rental_date
328,248,CAROLINE BOWMAN,336,FRENCH HOLIDAY,7.99,2005-05-27 02:15:30
616,248,CAROLINE BOWMAN,972,WHISPERER GIANT,4.99,2005-05-28 15:50:07
2064,248,CAROLINE BOWMAN,561,MASK PEACH,3.99,2005-06-17 16:07:08
2368,248,CAROLINE BOWMAN,39,ARMAGEDDON LOST,0.99,2005-06-18 14:35:29
3907,248,CAROLINE BOWMAN,452,ILLUSION AMELIE,0.99,2005-07-06 20:05:18
4538,248,CAROLINE BOWMAN,431,HOOSIERS BIRDCAGE,4.99,2005-07-08 04:04:19
4838,248,CAROLINE BOWMAN,114,CAMELOT VACATION,0.99,2005-07-08 18:18:23
5367,248,CAROLINE BOWMAN,474,JADE BUNCH,2.99,2005-07-09 18:43:19
6613,248,CAROLINE BOWMAN,498,KILLER INNOCENT,2.99,2005-07-12 08:39:56
7774,248,CAROLINE BOWMAN,233,DISCIPLE MOTHER,5.99,2005-07-28 07:10:11


## Soru 42: En az kazanç sağlayan müşteri hangi türde en fazla film kiralamış?


In [45]:
low_customer_id = rental_full.groupby("customer_id")["amount"].sum().idxmin()
soru42 = (
    rental_full[rental_full["customer_id"] == low_customer_id]
    .groupby(["customer_id", "customer_name", "category_name"])["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
    .sort_values("rental_count", ascending=False)
    .head(1)
)
soru42


,customer_id,customer_name,category_name,rental_count
6,248,CAROLINE BOWMAN,Sci-Fi,4


## Soru 43: En çok kiralanan film hangi çalışan tarafından kiralanmış?


In [ ]:
top_film_id = rental_full.groupby("film_id")["rental_id"].nunique().idxmax()
soru43 = (
    rental_full[rental_full["film_id"] == top_film_id]
    .groupby(["film_id", "title", "staff_id", "staff_name"])["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
    .sort_values("rental_count", ascending=False)
)
soru43


## Soru 44: En az kiralanan film hangi çalışan tarafından kiralanmış?


In [46]:
film_counts = rental_full.groupby("film_id")["rental_id"].nunique()
low_film_id = film_counts.idxmin()
soru44 = (
    rental_full[rental_full["film_id"] == low_film_id]
    .groupby(["film_id", "title", "staff_id", "staff_name"])["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
    .sort_values("rental_count", ascending=False)
)
soru44


,film_id,title,staff_id,staff_name,rental_count
1,400,HARDLY ROBBERS,2,Jon Stephens,3
0,400,HARDLY ROBBERS,1,Mike Hillyer,1


## Soru 45: En çok kazanç sağlayan film hangi çalışan tarafından kiralanmış?


In [47]:
top_film_id = rental_full.groupby("film_id")["amount"].sum().idxmax()
soru45 = (
    rental_full[rental_full["film_id"] == top_film_id]
    .groupby(["film_id", "title", "staff_id", "staff_name"])
    .agg(rental_count=("rental_id", "nunique"), total_revenue=("amount", "sum"))
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
soru45


,film_id,title,staff_id,staff_name,rental_count,total_revenue
1,879,TELEGRAPH VOYAGE,2,Jon Stephens,17,147.83
0,879,TELEGRAPH VOYAGE,1,Mike Hillyer,10,83.90


## Soru 46: En az kazanç sağlayan film hangi çalışan tarafından kiralanmış?


In [48]:
low_film_id = rental_full.groupby("film_id")["amount"].sum().idxmin()
soru46 = (
    rental_full[rental_full["film_id"] == low_film_id]
    .groupby(["film_id", "title", "staff_id", "staff_name"])
    .agg(rental_count=("rental_id", "nunique"), total_revenue=("amount", "sum"))
    .reset_index()
    .sort_values("total_revenue", ascending=True)
)
soru46


,film_id,title,staff_id,staff_name,rental_count,total_revenue
1,635,OKLAHOMA JUMANJI,2,Jon Stephens,2,1.98
0,635,OKLAHOMA JUMANJI,1,Mike Hillyer,4,3.96


## Soru 47: En çok kiralanan film hangi mağazada kiralanmış?


In [49]:
top_film_id = rental_full.groupby("film_id")["rental_id"].nunique().idxmax()
soru47 = (
    rental_full[rental_full["film_id"] == top_film_id]
    .groupby(["film_id", "title", "store_id", "store_city"])["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
    .sort_values("rental_count", ascending=False)
)
soru47


,film_id,title,store_id,store_city,rental_count
0,103,BUCKET BROTHERHOOD,1,Lethbridge,17
1,103,BUCKET BROTHERHOOD,2,Woodridge,17


## Soru 48: En az kiralanan film hangi mağazada kiralanmış?


In [50]:
low_film_id = rental_full.groupby("film_id")["rental_id"].nunique().idxmin()
soru48 = (
    rental_full[rental_full["film_id"] == low_film_id]
    .groupby(["film_id", "title", "store_id", "store_city"])["rental_id"]
    .nunique()
    .reset_index(name="rental_count")
    .sort_values("rental_count", ascending=True)
)
soru48


,film_id,title,store_id,store_city,rental_count
0,400,HARDLY ROBBERS,1,Lethbridge,4


## Soru 49: En çok kazanç sağlayan film hangi mağazada kiralanmış?


In [51]:
top_film_id = rental_full.groupby("film_id")["amount"].sum().idxmax()
soru49 = (
    rental_full[rental_full["film_id"] == top_film_id]
    .groupby(["film_id", "title", "store_id", "store_city"])["amount"]
    .sum()
    .reset_index(name="total_revenue")
    .sort_values("total_revenue", ascending=False)
)
soru49


,film_id,title,store_id,store_city,total_revenue
0,879,TELEGRAPH VOYAGE,1,Lethbridge,132.85
1,879,TELEGRAPH VOYAGE,2,Woodridge,98.88


## Soru 50: En az kazanç sağlayan film hangi mağazada kiralanmış?


In [52]:
low_film_id = rental_full.groupby("film_id")["amount"].sum().idxmin()
soru50 = (
    rental_full[rental_full["film_id"] == low_film_id]
    .groupby(["film_id", "title", "store_id", "store_city"])["amount"]
    .sum()
    .reset_index(name="total_revenue")
    .sort_values("total_revenue", ascending=True)
)
soru50


,film_id,title,store_id,store_city,total_revenue
0,635,OKLAHOMA JUMANJI,2,Woodridge,5.94


## Soru 51: Müşterilerin kiraladıkları filmlerin toplam kiralama süresi ne kadar?


In [53]:
soru51 = (
    rental_full.dropna(subset=["rental_days"])
    .groupby(["customer_id", "customer_name"])["rental_days"]
    .sum()
    .reset_index(name="total_rental_days")
    .sort_values("total_rental_days", ascending=False)
)
soru51


,customer_id,customer_name,total_rental_days
525,526,KARL SEAL,264.151389
147,148,ELEANOR HUNT,243.095139
143,144,CLARA SHAW,235.040278
136,137,RHONDA KENNEDY,229.187500
294,295,DAISY BATES,221.306250
...,...,...,...
247,248,CAROLINE BOWMAN,72.142361
394,395,JOHNNY TURPIN,69.217361
96,97,ANNIE RUSSELL,64.425694
317,318,BRIAN WYMAN,59.231944


## Soru 52: En çok kiralanan türdeki filmler hangileri?


In [54]:
top_category = rental_full.groupby("category_name")["rental_id"].nunique().idxmax()
soru52 = film_cat[film_cat["category_name"] == top_category][["category_name", "film_id", "title"]].sort_values("title")
soru52


,category_name,film_id,title
9,Sports,10,ALADDIN CALENDAR
26,Sports,27,ANONYMOUS HUMAN
41,Sports,42,ARTIST COLDBLOODED
101,Sports,102,BUBBLE GROSSE
112,Sports,113,CALIFORNIA BIRDS
...,...,...,...
889,Sports,890,TIGHTS DAWN
897,Sports,898,TOURIST PELICAN
901,Sports,902,TRADING PINOCCHIO
916,Sports,917,TUXEDO MILE


## Soru 53: En az kiralanan türdeki filmler hangileri?


In [55]:
low_category = rental_full.groupby("category_name")["rental_id"].nunique().idxmin()
soru53 = film_cat[film_cat["category_name"] == low_category][["category_name", "film_id", "title"]].sort_values("title")
soru53


,category_name,film_id,title
11,Music,12,ALASKA PHANTOM
16,Music,17,ALONE TRIP
19,Music,20,AMELIE HELLFIGHTERS
50,Music,51,BALLOON HOMEWARD
53,Music,54,BANGER PINOCCHIO
73,Music,74,BIRCH ANTITRUST
75,Music,76,BIRDCAGE CASPER
85,Music,86,BOOGIE AMELIE
132,Music,133,CHAMBER ITALIAN
157,Music,158,CLONES PINOCCHIO


## Soru 54: Her müşteri için toplam ödeme miktarını bulun.


In [56]:
soru54 = (
    rental_full.groupby(["customer_id", "customer_name"])["amount"]
    .sum().reset_index(name="total_payment")
    .sort_values("total_payment", ascending=False)
)
soru54


,customer_id,customer_name,total_payment
525,526,KARL SEAL,221.55
147,148,ELEANOR HUNT,216.54
143,144,CLARA SHAW,195.58
136,137,RHONDA KENNEDY,194.61
177,178,MARION SNYDER,194.61
...,...,...,...
96,97,ANNIE RUSSELL,58.82
394,395,JOHNNY TURPIN,57.81
317,318,BRIAN WYMAN,52.88
280,281,LEONA OBRIEN,50.86


## Soru 55: Hangi filmler en uzun süre kiralanmış?


In [57]:
soru55 = (
    rental_full.dropna(subset=["rental_days"])
    .groupby(["film_id", "title"])["rental_days"]
    .max().reset_index(name="max_rental_days")
    .sort_values("max_rental_days", ascending=False)
)
soru55


,film_id,title,max_rental_days
400,424,HOLOCAUST HIGHBALL,9.249306
393,416,HIGHBALL POTTER,9.249306
160,172,CONEHEADS SMOOCHY,9.248611
625,653,PANIC CLUB,9.248611
535,561,MASK PEACH,9.248611
...,...,...,...
800,836,SQUAD FISH,5.908333
168,180,CONSPIRACY SPIRIT,5.809722
341,362,GLORY TRACY,5.120833
865,903,TRAFFIC HOBBIT,4.977778


## Soru 56: Hangi filmler en kısa süre kiralanmış?


In [58]:
soru56 = (
    rental_full.dropna(subset=["rental_days"])
    .groupby(["film_id", "title"])["rental_days"]
    .min().reset_index(name="min_rental_days")
    .sort_values("min_rental_days", ascending=True)
)
soru56


,film_id,title,min_rental_days
216,233,DISCIPLE MOTHER,0.750000
185,200,CURTAIN VIDEOTAPE,0.750000
701,734,ROAD ROXANNE,0.750694
815,851,STRAIGHT HOURS,0.751389
714,748,RUGRATS SHAKESPEARE,0.751389
...,...,...,...
558,584,MIXED DOORS,3.240972
251,268,EARLY HOME,3.807639
626,654,PANKY SUBMARINE,3.827083
911,952,WAGON JAWS,3.859028


## Soru 57: Müşterilerin kiraladığı filmler için ortalama ödeme miktarını bulun.


In [ ]:
soru57 = (
    rental_full.groupby(["customer_id", "customer_name"])["amount"]
    .mean().reset_index(name="avg_payment")
    .sort_values("avg_payment", ascending=False)
)
soru57


## Soru 58: En çok kiralanan filmlerin ortalama kiralama süresi nedir?


In [ ]:
top_films = rental_full.groupby(["film_id", "title"])["rental_id"].nunique().reset_index(name="rental_count").sort_values("rental_count", ascending=False).head(5)
soru58 = top_films.merge(
    rental_full.groupby("film_id")["rental_days"].mean().reset_index(name="avg_rental_days"),
    on="film_id", how="left"
)
soru58


## Soru 59: En az kiralanan filmlerin ortalama kiralama süresi nedir?


In [59]:
low_films = rental_full.groupby(["film_id", "title"])["rental_id"].nunique().reset_index(name="rental_count").sort_values("rental_count", ascending=True).head(5)
soru59 = low_films.merge(
    rental_full.groupby("film_id")["rental_days"].mean().reset_index(name="avg_rental_days"),
    on="film_id", how="left"
)
soru59


,film_id,title,rental_count,avg_rental_days
0,584,MIXED DOORS,4,5.953299
1,904,TRAIN BUNCH,4,3.349306
2,400,HARDLY ROBBERS,4,6.948438
3,612,MUSSOLINI SPOILERS,5,4.038611
4,335,FREEDOM CLEOPATRA,5,2.959722


## Soru 60: En çok kazanç sağlayan müşterilerin ortalama ödeme miktarı nedir?


In [60]:
top_customers = rental_full.groupby(["customer_id", "customer_name"])["amount"].sum().reset_index(name="total_revenue").sort_values("total_revenue", ascending=False).head(5)
soru60 = top_customers.merge(
    rental_full.groupby("customer_id")["amount"].mean().reset_index(name="avg_payment"),
    on="customer_id", how="left"
)
soru60


,customer_id,customer_name,total_revenue,avg_payment
0,526,KARL SEAL,221.55,4.923333
1,148,ELEANOR HUNT,216.54,4.707391
2,144,CLARA SHAW,195.58,4.656667
3,137,RHONDA KENNEDY,194.61,4.990000
4,178,MARION SNYDER,194.61,4.990000


## Soru 61: En az kazanç sağlayan müşterilerin ortalama ödeme miktarı nedir?


In [61]:
low_customers = rental_full.groupby(["customer_id", "customer_name"])["amount"].sum().reset_index(name="total_revenue").sort_values("total_revenue", ascending=True).head(5)
soru61 = low_customers.merge(
    rental_full.groupby("customer_id")["amount"].mean().reset_index(name="avg_payment"),
    on="customer_id", how="left"
)
soru61


,customer_id,customer_name,total_revenue,avg_payment
0,248,CAROLINE BOWMAN,50.85,3.390000
1,281,LEONA OBRIEN,50.86,3.632857
2,318,BRIAN WYMAN,52.88,4.406667
3,395,JOHNNY TURPIN,57.81,3.042632
4,97,ANNIE RUSSELL,58.82,3.267778


## Soru 62: Mağazalardaki toplam kiralama süresini bulun.


In [62]:
soru62 = (
    rental_full.dropna(subset=["rental_days"])
    .groupby(["store_id", "store_city"])["rental_days"]
    .sum().reset_index(name="total_rental_days")
    .sort_values("total_rental_days", ascending=False)
)
soru62


,store_id,store_city,total_rental_days
1,2,Woodridge,40267.343056
0,1,Lethbridge,39439.421528


## Soru 63: En uzun süre kiralanan film hangi mağazada kiralanmış?


In [63]:
soru63 = rental_full.dropna(subset=["rental_days"]).sort_values("rental_days", ascending=False)[["film_id", "title", "store_id", "store_city", "rental_days"]].head(1)
soru63


,film_id,title,store_id,store_city,rental_days
14673,416,HIGHBALL POTTER,1,Lethbridge,9.249306


## Soru 64: En kısa süre kiralanan film hangi mağazada kiralanmış?


In [64]:
soru64 = rental_full.dropna(subset=["rental_days"]).sort_values("rental_days", ascending=True)[["film_id", "title", "store_id", "store_city", "rental_days"]].head(1)
soru64


,film_id,title,store_id,store_city,rental_days
10297,233,DISCIPLE MOTHER,1,Lethbridge,0.75


## Soru 65: Her filmin ortalama kiralama süresi nedir?


In [65]:
soru65 = rental_full.dropna(subset=["rental_days"]).groupby(["film_id", "title"])["rental_days"].mean().reset_index(name="avg_rental_days").sort_values("avg_rental_days", ascending=False)
soru65


,film_id,title,avg_rental_days
305,323,FLIGHT LIES,7.270775
430,454,IMPACT ALADDIN,7.200617
4,5,AFRICAN EGG,7.106629
378,400,HARDLY ROBBERS,6.948438
520,546,MADRE GABLES,6.811806
...,...,...,...
847,885,TEXAS WATCH,3.016667
754,788,SHIP WONDERLAND,2.986296
315,335,FREEDOM CLEOPATRA,2.959722
453,477,JAWBREAKER BROOKLYN,2.585764


## Soru 66: Hangi filmler en çok kiralanan kategoride?


In [66]:
top_category = rental_full.groupby("category_name")["rental_id"].nunique().idxmax()
soru66 = film_cat[film_cat["category_name"] == top_category][["category_name", "film_id", "title"]].sort_values("title")
soru66


,category_name,film_id,title
9,Sports,10,ALADDIN CALENDAR
26,Sports,27,ANONYMOUS HUMAN
41,Sports,42,ARTIST COLDBLOODED
101,Sports,102,BUBBLE GROSSE
112,Sports,113,CALIFORNIA BIRDS
...,...,...,...
889,Sports,890,TIGHTS DAWN
897,Sports,898,TOURIST PELICAN
901,Sports,902,TRADING PINOCCHIO
916,Sports,917,TUXEDO MILE


## Soru 67: Hangi filmler en az kiralanan kategoride?


In [67]:
low_category = rental_full.groupby("category_name")["rental_id"].nunique().idxmin()
soru67 = film_cat[film_cat["category_name"] == low_category][["category_name", "film_id", "title"]].sort_values("title")
soru67


,category_name,film_id,title
11,Music,12,ALASKA PHANTOM
16,Music,17,ALONE TRIP
19,Music,20,AMELIE HELLFIGHTERS
50,Music,51,BALLOON HOMEWARD
53,Music,54,BANGER PINOCCHIO
73,Music,74,BIRCH ANTITRUST
75,Music,76,BIRDCAGE CASPER
85,Music,86,BOOGIE AMELIE
132,Music,133,CHAMBER ITALIAN
157,Music,158,CLONES PINOCCHIO


## Soru 68: Hangi mağazalarda en çok film kiralanmış?


In [68]:
soru68 = rental_full.groupby(["store_id", "store_city"])["rental_id"].nunique().reset_index(name="rental_count").sort_values("rental_count", ascending=False)
soru68


,store_id,store_city,rental_count
1,2,Woodridge,8121
0,1,Lethbridge,7923


## Soru 69: Hangi mağazalarda en az film kiralanmış?


In [69]:
soru69 = rental_full.groupby(["store_id", "store_city"])["rental_id"].nunique().reset_index(name="rental_count").sort_values("rental_count", ascending=True)
soru69


,store_id,store_city,rental_count
0,1,Lethbridge,7923
1,2,Woodridge,8121


## Soru 70: Hangi aktörler en çok filmde rol almış?


In [70]:
soru70 = actor_df[["actor_id", "actor_name"]].merge(film_actor, on="actor_id").groupby(["actor_id", "actor_name"])["film_id"].nunique().reset_index(name="film_count").sort_values("film_count", ascending=False)
soru70


,actor_id,actor_name,film_count
106,107,GINA DEGENERES,42
101,102,WALTER TORN,41
197,198,MARY KEITEL,40
180,181,MATTHEW CARREY,39
22,23,SANDRA KILMER,37
...,...,...,...
70,71,ADAM GRANT,18
185,186,JULIA ZELLWEGER,16
34,35,JUDY DEAN,15
198,199,JULIA FAWCETT,15


## Soru 71: Hangi aktörler en az filmde rol almış?


In [71]:
soru71 = actor_df[["actor_id", "actor_name"]].merge(film_actor, on="actor_id").groupby(["actor_id", "actor_name"])["film_id"].nunique().reset_index(name="film_count").sort_values("film_count", ascending=True)
soru71


,actor_id,actor_name,film_count
147,148,EMILY DEE,14
198,199,JULIA FAWCETT,15
34,35,JUDY DEAN,15
185,186,JULIA ZELLWEGER,16
70,71,ADAM GRANT,18
...,...,...,...
22,23,SANDRA KILMER,37
180,181,MATTHEW CARREY,39
197,198,MARY KEITEL,40
101,102,WALTER TORN,41


## Soru 72: Hangi aktörlerin oynadığı filmler en çok kiralanmış?


In [72]:
actor_rentals = film_actor.merge(actor_df[["actor_id", "actor_name"]], on="actor_id").merge(rental_full[["film_id", "rental_id"]], on="film_id", how="left")
soru72 = actor_rentals.groupby(["actor_id", "actor_name"])["rental_id"].nunique().reset_index(name="rental_count").sort_values("rental_count", ascending=False)
soru72


,actor_id,actor_name,rental_count
106,107,GINA DEGENERES,753
180,181,MATTHEW CARREY,678
197,198,MARY KEITEL,674
143,144,ANGELA WITHERSPOON,654
101,102,WALTER TORN,640
...,...,...,...
34,35,JUDY DEAN,255
198,199,JULIA FAWCETT,255
30,31,SISSY SOBIESKI,235
185,186,JULIA ZELLWEGER,221


## Soru 73: Hangi aktörlerin oynadığı filmler en az kiralanmış?


In [73]:
actor_rentals = film_actor.merge(actor_df[["actor_id", "actor_name"]], on="actor_id").merge(rental_full[["film_id", "rental_id"]], on="film_id", how="left")
soru73 = actor_rentals.groupby(["actor_id", "actor_name"])["rental_id"].nunique().reset_index(name="rental_count").sort_values("rental_count", ascending=True)
soru73


,actor_id,actor_name,rental_count
147,148,EMILY DEE,216
185,186,JULIA ZELLWEGER,221
30,31,SISSY SOBIESKI,235
34,35,JUDY DEAN,255
198,199,JULIA FAWCETT,255
...,...,...,...
101,102,WALTER TORN,640
143,144,ANGELA WITHERSPOON,654
197,198,MARY KEITEL,674
180,181,MATTHEW CARREY,678


## Soru 74: Hangi kategorilerde en fazla aktör oynamış?


In [74]:
cat_actor = film_category.merge(category.rename(columns={"name": "category_name"})[["category_id", "category_name"]], on="category_id").merge(film_actor, on="film_id")
soru74 = cat_actor.groupby("category_name")["actor_id"].nunique().reset_index(name="actor_count").sort_values("actor_count", ascending=False)
soru74


,category_name,actor_count
14,Sports,182
8,Foreign,175
12,New,169
5,Documentary,168
13,Sci-Fi,167
0,Action,166
1,Animation,166
15,Travel,166
7,Family,164
2,Children,163


## Soru 75: Hangi kategorilerde en az aktör oynamış?


In [75]:
cat_actor = film_category.merge(category.rename(columns={"name": "category_name"})[["category_id", "category_name"]], on="category_id").merge(film_actor, on="film_id")
soru75 = cat_actor.groupby("category_name")["actor_id"].nunique().reset_index(name="actor_count").sort_values("actor_count", ascending=True)
soru75


,category_name,actor_count
11,Music,144
4,Comedy,147
9,Games,150
10,Horror,156
3,Classics,162
6,Drama,162
2,Children,163
7,Family,164
0,Action,166
1,Animation,166


## Soru 76: Hangi kategorilerde oynayan filmler en çok kiralanmış?


In [76]:
soru76 = rental_full.groupby("category_name")["rental_id"].nunique().reset_index(name="rental_count").sort_values("rental_count", ascending=False)
soru76


,category_name,rental_count
14,Sports,1179
1,Animation,1166
0,Action,1112
13,Sci-Fi,1101
7,Family,1096
6,Drama,1060
5,Documentary,1050
8,Foreign,1033
9,Games,969
2,Children,945


## Soru 77: Hangi kategorilerde oynayan filmler en az kiralanmış?


In [77]:
soru77 = rental_full.groupby("category_name")["rental_id"].nunique().reset_index(name="rental_count").sort_values("rental_count", ascending=True)
soru77


,category_name,rental_count
11,Music,830
15,Travel,837
10,Horror,846
3,Classics,939
12,New,940
4,Comedy,941
2,Children,945
9,Games,969
8,Foreign,1033
5,Documentary,1050


## Soru 78: En fazla kazanç sağlayan kategoriler hangileri?


In [78]:
soru78 = rental_full.groupby("category_name")["amount"].sum().reset_index(name="total_revenue").sort_values("total_revenue", ascending=False)
soru78


,category_name,total_revenue
14,Sports,5314.21
13,Sci-Fi,4756.98
1,Animation,4656.30
6,Drama,4587.39
4,Comedy,4383.58
0,Action,4375.85
12,New,4351.62
9,Games,4281.33
8,Foreign,4270.67
7,Family,4226.07


## Soru 79: En az kazanç sağlayan kategoriler hangileri?


In [79]:
soru79 = rental_full.groupby("category_name")["amount"].sum().reset_index(name="total_revenue").sort_values("total_revenue", ascending=True)
soru79


,category_name,total_revenue
11,Music,3417.72
15,Travel,3549.64
3,Classics,3639.59
2,Children,3655.55
10,Horror,3722.54
5,Documentary,4217.52
7,Family,4226.07
8,Foreign,4270.67
9,Games,4281.33
12,New,4351.62


## Soru 80: En çok film kiralayan müşterilerin ortalama ödeme miktarı nedir?


In [80]:
top_customers = rental_full.groupby(["customer_id", "customer_name"])["rental_id"].nunique().reset_index(name="rental_count").sort_values("rental_count", ascending=False).head(5)
soru80 = top_customers.merge(rental_full.groupby("customer_id")["amount"].mean().reset_index(name="avg_payment"), on="customer_id", how="left")
soru80


,customer_id,customer_name,rental_count,avg_payment
0,148,ELEANOR HUNT,46,4.707391
1,526,KARL SEAL,45,4.923333
2,144,CLARA SHAW,42,4.656667
3,236,MARCIA DEAN,42,4.180476
4,75,TAMMY SANDERS,41,3.794878


## Soru 81: En az film kiralayan müşterilerin ortalama ödeme miktarı nedir?


In [81]:
low_customers = rental_full.groupby(["customer_id", "customer_name"])["rental_id"].nunique().reset_index(name="rental_count").sort_values("rental_count", ascending=True).head(5)
soru81 = low_customers.merge(rental_full.groupby("customer_id")["amount"].mean().reset_index(name="avg_payment"), on="customer_id", how="left")
soru81


,customer_id,customer_name,rental_count,avg_payment
0,318,BRIAN WYMAN,12,4.406667
1,281,LEONA OBRIEN,14,3.632857
2,110,TIFFANY JORDAN,14,4.275714
3,61,KATHERINE RIVERA,14,4.204286
4,136,ANITA MORALES,15,4.190000


## Soru 82: Hangi mağazalarda hangi kategoriler en çok kiralanmış?


In [82]:
tmp = rental_full.groupby(["store_id", "store_city", "category_name"])["rental_id"].nunique().reset_index(name="rental_count")
soru82 = tmp.loc[tmp.groupby("store_id")["rental_count"].idxmax()].sort_values("store_id")
soru82


,store_id,store_city,category_name,rental_count
0,1,Lethbridge,Action,596
30,2,Woodridge,Sports,624


## Soru 83: Hangi mağazalarda hangi kategoriler en az kiralanmış?


In [83]:
tmp = rental_full.groupby(["store_id", "store_city", "category_name"])["rental_id"].nunique().reset_index(name="rental_count")
soru83 = tmp.loc[tmp.groupby("store_id")["rental_count"].idxmin()].sort_values("store_id")
soru83


,store_id,store_city,category_name,rental_count
10,1,Lethbridge,Horror,386
27,2,Woodridge,Music,394


## Soru 84: En fazla sayıda film kiralayan mağaza hangisidir ve toplam kiralama sayısı nedir?


In [84]:
soru84 = rental_full.groupby(["store_id", "store_city"])["rental_id"].nunique().reset_index(name="total_rentals").sort_values("total_rentals", ascending=False).head(1)
soru84


,store_id,store_city,total_rentals
1,2,Woodridge,8121


## Soru 85: En az sayıda film kiralayan mağaza hangisidir ve toplam kiralama sayısı nedir?


In [85]:
soru85 = rental_full.groupby(["store_id", "store_city"])["rental_id"].nunique().reset_index(name="total_rentals").sort_values("total_rentals", ascending=True).head(1)
soru85


,store_id,store_city,total_rentals
0,1,Lethbridge,7923


## Soru 86: En fazla sayıda kiralama yapan müşteri hangisidir ve toplam kiralama sayısı nedir?


In [86]:
soru86 = rental_full.groupby(["customer_id", "customer_name"])["rental_id"].nunique().reset_index(name="total_rentals").sort_values("total_rentals", ascending=False).head(1)
soru86


,customer_id,customer_name,total_rentals
147,148,ELEANOR HUNT,46


## Soru 87: En az sayıda kiralama yapan müşteri hangisidir ve toplam kiralama sayısı nedir?


In [87]:
soru87 = rental_full.groupby(["customer_id", "customer_name"])["rental_id"].nunique().reset_index(name="total_rentals").sort_values("total_rentals", ascending=True).head(1)
soru87


,customer_id,customer_name,total_rentals
317,318,BRIAN WYMAN,12


## Soru 88: En fazla sayıda kiralanan film hangisidir ve toplam kiralama sayısı nedir?


In [88]:
soru88 = rental_full.groupby(["film_id", "title"])["rental_id"].nunique().reset_index(name="total_rentals").sort_values("total_rentals", ascending=False).head(1)
soru88


,film_id,title,total_rentals
96,103,BUCKET BROTHERHOOD,34


## Soru 89: En az sayıda kiralanan film hangisidir ve toplam kiralama sayısı nedir?


In [89]:
soru89 = rental_full.groupby(["film_id", "title"])["rental_id"].nunique().reset_index(name="total_rentals").sort_values("total_rentals", ascending=True).head(1)
soru89


,film_id,title,total_rentals
558,584,MIXED DOORS,4


## Soru 90: En fazla kazanç sağlayan müşteri hangisidir ve toplam kazancı nedir?


In [90]:
soru90 = rental_full.groupby(["customer_id", "customer_name"])["amount"].sum().reset_index(name="total_revenue").sort_values("total_revenue", ascending=False).head(1)
soru90


,customer_id,customer_name,total_revenue
525,526,KARL SEAL,221.55


## Soru 91: En az kazanç sağlayan müşteri hangisidir ve toplam kazancı nedir?


In [91]:
soru91 = rental_full.groupby(["customer_id", "customer_name"])["amount"].sum().reset_index(name="total_revenue").sort_values("total_revenue", ascending=True).head(1)
soru91


,customer_id,customer_name,total_revenue
247,248,CAROLINE BOWMAN,50.85


## Soru 92: Hangi film, hangi kategoride en çok kazanç sağlamış?


In [92]:
soru92 = rental_full.groupby(["film_id", "title", "category_name"])["amount"].sum().reset_index(name="total_revenue").sort_values("total_revenue", ascending=False).head(1)
soru92


,film_id,title,category_name,total_revenue
841,879,TELEGRAPH VOYAGE,Music,231.73


## Soru 93: Hangi film, hangi kategoride en az kazanç sağlamış?


In [93]:
soru93 = rental_full.groupby(["film_id", "title", "category_name"])["amount"].sum().reset_index(name="total_revenue").sort_values("total_revenue", ascending=True).head(1)
soru93


,film_id,title,category_name,total_revenue
847,885,TEXAS WATCH,Horror,5.94


## Soru 94: En fazla kazanç sağlayan mağaza hangisidir ve toplam kazancı nedir?


In [94]:
soru94 = rental_full.groupby(["store_id", "store_city"])["amount"].sum().reset_index(name="total_revenue").sort_values("total_revenue", ascending=False).head(1)
soru94


,store_id,store_city,total_revenue
1,2,Woodridge,33726.77


## Soru 95: En az kazanç sağlayan mağaza hangisidir ve toplam kazancı nedir?


In [95]:
soru95 = rental_full.groupby(["store_id", "store_city"])["amount"].sum().reset_index(name="total_revenue").sort_values("total_revenue", ascending=True).head(1)
soru95


,store_id,store_city,total_revenue
0,1,Lethbridge,33679.79


## Soru 96: En fazla sayıda farklı film kiralayan müşteri hangisidir?


In [96]:
soru96 = rental_full.groupby(["customer_id", "customer_name"])["film_id"].nunique().reset_index(name="distinct_film_count").sort_values("distinct_film_count", ascending=False).head(1)
soru96


,customer_id,customer_name,distinct_film_count
147,148,ELEANOR HUNT,46


## Soru 97: En az sayıda farklı film kiralayan müşteri hangisidir?


In [97]:
soru97 = rental_full.groupby(["customer_id", "customer_name"])["film_id"].nunique().reset_index(name="distinct_film_count").sort_values("distinct_film_count", ascending=True).head(1)
soru97


,customer_id,customer_name,distinct_film_count
317,318,BRIAN WYMAN,12


## Soru 98: En fazla sayıda farklı müşteri tarafından kiralanan film hangisidir?


In [98]:
soru98 = rental_full.groupby(["film_id", "title"])["customer_id"].nunique().reset_index(name="distinct_customer_count").sort_values("distinct_customer_count", ascending=False).head(1)
soru98


,film_id,title,distinct_customer_count
96,103,BUCKET BROTHERHOOD,33


## Soru 99: En az sayıda farklı müşteri tarafından kiralanan film hangisidir?


In [99]:
soru99 = rental_full.groupby(["film_id", "title"])["customer_id"].nunique().reset_index(name="distinct_customer_count").sort_values("distinct_customer_count", ascending=True).head(1)
soru99


,film_id,title,distinct_customer_count
866,904,TRAIN BUNCH,4


## Soru 100: Hangi mağazada, hangi kategoride en fazla kazanç sağlanmış?


In [100]:
soru100 = rental_full.groupby(["store_id", "store_city", "category_name"])["amount"].sum().reset_index(name="total_revenue").sort_values("total_revenue", ascending=False).head(1)
soru100


,store_id,store_city,category_name,total_revenue
30,2,Woodridge,Sports,2825.75
